In [ ]:
# Cell 1: imports and paths configuration

import os
import numpy as np
import pandas as pd
from typing import Tuple

# Setup paths
PROJECT_DIR = os.path.dirname(os.path.abspath("__file__"))  # fallback se __file__ non esiste
BASE = os.path.join(PROJECT_DIR, "POLITISKY24_Layers")

REPLY_PATH = os.path.join(BASE, "signed_layers", "edges_reply_signed.parquet")
QUOTE_PATH = os.path.join(BASE, "signed_layers", "edges_quote_signed.parquet")
LIKES_PATH = os.path.join(BASE, "likes.parquet")
REPOST_PATH = os.path.join(BASE, "reposts.parquet")

OUT_DIR = os.path.join(BASE, "cumulative_signed_layers")
os.makedirs(OUT_DIR, exist_ok=True)

PARQUET_KW = dict(engine="pyarrow")

In [2]:
# Load dataframes
df_reply = pd.read_parquet(REPLY_PATH, **PARQUET_KW)
df_quote = pd.read_parquet(QUOTE_PATH, **PARQUET_KW)
df_likes = pd.read_parquet(LIKES_PATH, **PARQUET_KW)
df_repost = pd.read_parquet(REPOST_PATH, **PARQUET_KW)

# Debug print
print("Reply head:")
print(df_reply.head())
print("\nQuote head:")
print(df_quote.head())
print("\nLikes head:")
print(df_likes.head())
print("\nRepost head:")
print(df_repost.head())

Reply head:
   ReplyPostId  ParentPostId  ReplierId  ParentUserId  \
0      1684791      10321242      84980        227274   
1      1359450       8522895     102066        184624   
2      8474276       2718283     346452         60001   
3     10421637      13562637     380231        297143   
4     12223962      14865619     272137        323191   

                                        ReplyContent  \
0                                 So good and funny!   
1  Love her! She never lets up on the lying hypoc...   
2  When I called one of my MAGA members of Congre...   
3  They should throw in something completely rand...   
4  I love this visual 💜\n\nI was looking through ...   

                                 OriginalPostContent   Type  Weight  P_contra  \
0  Signs you might be a far right authoritarian p...  Reply       1  0.014954   
1  Crockett: 19 of the 26 Republicans on this com...  Reply       1  0.020142   
2  🪖Women in the military will NOT be safe with P...  Reply      

## Merge results into a single network

In [ ]:
# === Layer weight parameters ===
# Weights assigned heuristically based on perceived importance
LAYER_WEIGHTS = {
    "like": 1.0,
    "repost": 1.0,
    "reply": 5.0,
    "quote": 5.0,
}

# Sign mapping for "signed" edges
# EdgeSign: 1 => +1 (entail/pro), 0 => -1 (contra)
SIGNED_MAP = {1:  +1, 0: -1}


In [4]:
import numpy as np
import pandas as pd

SIGNED_MAP = {1: +1, 0: -1}

def resolve_signed_col(d: pd.DataFrame) -> pd.Series:
    """
    Restituisce una Series int8 con il segno:
    - +1 se EdgeSign==1
    - -1 se EdgeSign==0
    - se EdgeSign è NaN, prova con P_entail vs P_contra
    - se ancora NaN, assegna 0
    """
    s = d["EdgeSign"].map(SIGNED_MAP) if "EdgeSign" in d.columns else pd.Series(index=d.index, dtype="float64")

    if "P_entail" in d.columns and "P_contra" in d.columns:
        # where s is NaN, deduce from probability comparison
        undecided = s.isna()
        if undecided.any():
            pe = d.loc[undecided, "P_entail"]
            pc = d.loc[undecided, "P_contra"]
            s.loc[undecided] = np.where(pe > pc, 1, np.where(pc > pe, -1, 0))

    s = s.fillna(0).astype("int8")
    return s

def _ensure_int(df: pd.DataFrame, cols):
    out = df.copy()
    for c in cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    # drop rows with missing source/target
    before = len(out)
    out = out.dropna(subset=list(cols))
    if len(out) < before:
        print(f"Dropped {before - len(out)} rows with missing user ids in {list(cols)}")
    out[cols[0]] = out[cols[0]].astype("int64")
    out[cols[1]] = out[cols[1]].astype("int64")
    return out

# ----- Reply: ReplierId -> ParentUserId -----
edges_reply = (
    df_reply
    .assign(
        source=lambda d: d["ReplierId"],
        target=lambda d: d["ParentUserId"],
        sign=lambda d: resolve_signed_col(d),
        base_w=lambda d: pd.to_numeric(d.get("Weight", 1), errors="coerce").fillna(1.0).astype("float64"),
        layer="reply",
    )[["source", "target", "sign", "base_w", "layer"]]
)
edges_reply = _ensure_int(edges_reply, ("source", "target"))

# ----- Quote: QuoterId -> QuotedUserId -----
edges_quote = (
    df_quote
    .assign(
        source=lambda d: d["QuoterId"],
        target=lambda d: d["QuotedUserId"],
        sign=lambda d: resolve_signed_col(d),
        base_w=lambda d: pd.to_numeric(d.get("Weight", 1), errors="coerce").fillna(1.0).astype("float64"),
        layer="quote",
    )[["source", "target", "sign", "base_w", "layer"]]
)
edges_quote = _ensure_int(edges_quote, ("source", "target"))

# ----- Like: SourceUserId -> TargetUserId (positive) -----
edges_like = (
    df_likes
    .assign(
        source=lambda d: d["SourceUserId"],
        target=lambda d: d["TargetUserId"],
        sign=1,  # sempre positivo
        base_w=lambda d: pd.to_numeric(d.get("Count", 1), errors="coerce").fillna(1.0).astype("float64"),
        layer="like",
    )[["source", "target", "sign", "base_w", "layer"]]
)
edges_like = _ensure_int(edges_like, ("source", "target"))

# ----- Repost: SourceUserId -> TargetUserId (positive) -----
edges_repost = (
    df_repost
    .assign(
        source=lambda d: d["SourceUserId"],
        target=lambda d: d["TargetUserId"],
        sign=1,  # sempre positivo
        base_w=lambda d: pd.to_numeric(d.get("Count", 1), errors="coerce").fillna(1.0).astype("float64"),
        layer="repost",
    )[["source", "target", "sign", "base_w", "layer"]]
)
edges_repost = _ensure_int(edges_repost, ("source", "target"))


In [5]:

# Stack all interactions in the same schema
edges_all = pd.concat([edges_reply, edges_quote, edges_like, edges_repost], ignore_index=True)

# Apply layer weight
edges_all["layer_w"] = edges_all["layer"].map(LAYER_WEIGHTS).astype("float64")
edges_all["signed_contrib"] = edges_all["sign"].astype("float64") * edges_all["base_w"] * edges_all["layer_w"]
edges_all["abs_contrib"] = edges_all["base_w"] * edges_all["layer_w"]

print("Schema unificato — preview:")
print(edges_all.head())

Schema unificato — preview:
   source  target  sign  base_w  layer  layer_w  signed_contrib  abs_contrib
0   84980  227274     1     1.0  reply      5.0             5.0          5.0
1  102066  184624    -1     1.0  reply      5.0            -5.0          5.0
2  346452   60001    -1     1.0  reply      5.0            -5.0          5.0
3  380231  297143    -1     1.0  reply      5.0            -5.0          5.0
4  272137  323191    -1     1.0  reply      5.0            -5.0          5.0


In [6]:
# Group and sum by (source, target) maintaining direction
agg = (
    edges_all
    .groupby(["source", "target"], as_index=False)
    .agg(
        signed_weight=("signed_contrib", "sum"),
        total_abs=("abs_contrib", "sum"),
        interactions=("base_w", "sum")
    )
)

# Edge sign: sign(signed_weight) with 0 if signed weight is 0
agg["edge_sign"] = agg["signed_weight"].apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0)).astype("int8")

# Sort by intensity (absolute value)
agg = agg.sort_values(by="signed_weight", ascending=True).reset_index(drop=True)

print("Aggregato — preview:")
print(agg.head(10))
print("\nDistribuzione segni:")
print(agg["edge_sign"].value_counts(dropna=False))


Aggregato — preview:
   source  target  signed_weight  total_abs  interactions  edge_sign
0  241259  260161        -7495.0     9795.0        1959.0         -1
1  187676  247716        -3240.0     4760.0        1060.0         -1
2  328550  283415        -2310.0     3250.0         650.0         -1
3  187676   31639        -2207.0     5233.0        1501.0         -1
4  273555  252897        -2170.0     2420.0         484.0         -1
5  114639   28498        -1760.0     1780.0         356.0         -1
6  169089  283415        -1610.0     2150.0         430.0         -1
7  294759  254867        -1585.0     2455.0         491.0         -1
8  294759  308578        -1545.0     2235.0         447.0         -1
9  231823  247716        -1531.0     5059.0        2123.0         -1

Distribuzione segni:
edge_sign
 1    1140855
-1     570843
 0      20806
Name: count, dtype: int64


In [7]:
FINAL_PARQUET = os.path.join(OUT_DIR, "cumulative_signed_edges.parquet")
FINAL_CSV = os.path.join(OUT_DIR, "cumulative_signed_edges.csv")

# Save only required columns + some useful metrics
cols_save = ["source", "target", "edge_sign", "signed_weight", "total_abs", "interactions"]
agg.to_parquet(FINAL_PARQUET, index=False, engine="pyarrow")
agg.to_csv(FINAL_CSV, index=False)

print("Salvato:")
print(" -", FINAL_PARQUET)
print(" -", FINAL_CSV)


Salvato:
 - c:\Users\jacop\OneDrive - Università della Calabria\LM Intelligenza Artificiale\2 - Secondo Anno\Analisi di Social Network e Media\Progetto ASNM\POLITISKY24_Layers\cumulative_signed_layers\cumulative_signed_edges.parquet
 - c:\Users\jacop\OneDrive - Università della Calabria\LM Intelligenza Artificiale\2 - Secondo Anno\Analisi di Social Network e Media\Progetto ASNM\POLITISKY24_Layers\cumulative_signed_layers\cumulative_signed_edges.csv
